# Bradford Bulls — Scene Clip Extraction

Automatically cuts a long match video into **closeup clips** — segments where sponsor logos are most clearly visible.

**Pipeline:**
1. PySceneDetect finds shot boundaries (camera cuts)
2. YOLO classifies each scene: closeup / wide
3. Merge adjacent scenes, drop clips that are too short
4. Preview clip list before cutting
5. FFmpeg cuts clips without re-encoding → save to Drive

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
!nvidia-smi
!nvcc --version 2>/dev/null | grep release || echo 'nvcc not found'

## 2. Install dependencies

In [ ]:
!pip install -q scenedetect[opencv] ultralytics
!ffmpeg -version 2>/dev/null | head -1 || apt-get install -qq ffmpeg

## 3. Mount Drive + configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# >>> UPDATE this path to your video on Drive <<<
VIDEO_PATH = '/content/drive/MyDrive/bradford_bulls/videos/M05_white_1080p.mp4'

# ── Closeup detection ─────────────────────────────────────────────────────────
CLOSEUP_THRESHOLD = 0.30   # person_height / frame_height — above this = closeup
YOLO_CONF         = 0.40   # YOLO person detection confidence

# ── Clip filtering ────────────────────────────────────────────────────────────
MIN_CLIP_DURATION = 2.0    # drop clips shorter than N seconds
GAP_MERGE         = 1.5    # merge adjacent scenes if gap < N seconds

# ── Output ────────────────────────────────────────────────────────────────────
stem       = Path(VIDEO_PATH).stem
OUTPUT_DIR = Path(f'/content/drive/MyDrive/bradford_bulls/clips/{stem}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video  : {VIDEO_PATH}')
print(f'Exists : {Path(VIDEO_PATH).exists()}')
print(f'Output : {OUTPUT_DIR}')

## 4. Scene detection — find shot boundaries

In [ ]:
from scenedetect import detect, AdaptiveDetector

print('Detecting scene boundaries...')
raw_scenes = detect(VIDEO_PATH, AdaptiveDetector())

# convert to seconds
scenes_sec = [(s.get_seconds(), e.get_seconds()) for s, e in raw_scenes]

print(f'Found {len(scenes_sec)} scenes')
print(f'Example: {scenes_sec[:5]}')

## 5. Classify each scene — closeup or wide?

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm.notebook import tqdm

detector = YOLO('yolo26n.pt')

def classify_scene(video_path, start_s, end_s, threshold, conf):
    """Sample the middle frame of a scene, return (is_closeup, max_person_h_frac, n_persons)."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    mid = (start_s + end_s) / 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(mid * fps))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return False, 0.0, 0

    H = frame.shape[0]
    results = detector(frame, verbose=False, conf=conf, classes=[0])
    persons = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            persons.append(y2 - y1)

    if not persons:
        return False, 0.0, 0
    max_h = max(persons) / H
    return max_h >= threshold, round(max_h, 3), len(persons)


print(f'Classifying {len(scenes_sec)} scenes...')
classified = []
cap_tmp = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap_tmp.get(cv2.CAP_PROP_FPS) or 30.0
cap_tmp.release()

for start, end in tqdm(scenes_sec, desc='classifying'):
    is_cu, max_h, n = classify_scene(VIDEO_PATH, start, end,
                                      CLOSEUP_THRESHOLD, YOLO_CONF)
    classified.append({
        'start': start, 'end': end,
        'duration': round(end - start, 2),
        'is_closeup': is_cu,
        'max_person_h': max_h,
        'n_persons': n,
    })

n_cu = sum(1 for c in classified if c['is_closeup'])
print(f'\nCloseup : {n_cu} / {len(classified)} scenes')
print(f'Wide    : {len(classified) - n_cu} scenes (will be dropped)')

## 6. Merge adjacent scenes + filter short clips

In [ ]:
def merge_and_filter(classified, gap_merge, min_duration):
    """Merge adjacent closeup scenes and drop clips shorter than min_duration."""
    closeups = [c for c in classified if c['is_closeup']]
    if not closeups:
        return []

    merged = []
    cur_start = closeups[0]['start']
    cur_end   = closeups[0]['end']
    cur_max_h = closeups[0]['max_person_h']

    for c in closeups[1:]:
        if c['start'] - cur_end <= gap_merge:
            cur_end   = c['end']
            cur_max_h = max(cur_max_h, c['max_person_h'])
        else:
            merged.append((cur_start, cur_end, cur_max_h))
            cur_start = c['start']
            cur_end   = c['end']
            cur_max_h = c['max_person_h']
    merged.append((cur_start, cur_end, cur_max_h))

    return [(s, e, mh) for s, e, mh in merged if e - s >= min_duration]


clips = merge_and_filter(classified, GAP_MERGE, MIN_CLIP_DURATION)
total_dur = sum(e - s for s, e, _ in clips)

print(f'Clips after merge + filter : {len(clips)}')
print(f'Total duration             : {total_dur/60:.1f} min')

## 7. Preview clip list — review before cutting

In [ ]:
import pandas as pd
from IPython.display import display

def fmt_time(s):
    m, sec = divmod(int(s), 60)
    return f'{m:02d}:{sec:02d}'

rows = []
for i, (start, end, max_h) in enumerate(clips):
    rows.append({
        'clip':          f'clip_{i+1:03d}',
        'start':         fmt_time(start),
        'end':           fmt_time(end),
        'duration(s)':   round(end - start, 1),
        'max_person_h':  f'{max_h:.0%}',
        'out_file':      f'clip_{i+1:03d}_{fmt_time(start).replace(":","-")}.mp4',
    })

df_clips = pd.DataFrame(rows)
display(df_clips)
print(f'\nTotal: {len(clips)} clips, {total_dur/60:.1f} min')

## 8. Visualize thumbnails

In [ ]:
import math
import matplotlib.pyplot as plt
import cv2

THUMB_COLS = 4
MAX_THUMBS = 24

show_clips = clips[:MAX_THUMBS]
n_rows = math.ceil(len(show_clips) / THUMB_COLS)

fig, axes = plt.subplots(n_rows, THUMB_COLS,
                         figsize=(THUMB_COLS * 4, n_rows * 2.8),
                         squeeze=False)
axes = axes.flatten()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

for i, (start, end, max_h) in enumerate(show_clips):
    mid = (start + end) / 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(mid * fps))
    ok, frame = cap.read()
    ax = axes[i]
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'clip_{i+1:03d}  {fmt_time(start)}–{fmt_time(end)}\n'
                 f'{end-start:.1f}s  person {max_h:.0%}', fontsize=7)
    ax.axis('off')

cap.release()
for i in range(len(show_clips), len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Closeup clips — {stem}  ({len(clips)} total)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clips_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Preview saved → {OUTPUT_DIR}/clips_preview.png')

## 9. Cut clips — ffmpeg (no re-encode, near realtime)

Review the thumbnails above before running this cell.

In [ ]:
import subprocess
from tqdm.notebook import tqdm

def cut_clip(src, start, end, out_path):
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start:.3f}',
        '-to', f'{end:.3f}',
        '-i', src,
        '-c', 'copy',      # no re-encode — preserves quality, runs at ~realtime speed
        str(out_path)
    ]
    result = subprocess.run(cmd, capture_output=True)
    return result.returncode == 0


print(f'Cutting {len(clips)} clips → {OUTPUT_DIR}\n')
n_ok = n_fail = 0

for i, (start, end, _) in enumerate(tqdm(clips, desc='cutting')):
    out_name = f'clip_{i+1:03d}_{fmt_time(start).replace(":","-")}.mp4'
    out_path  = OUTPUT_DIR / out_name
    ok = cut_clip(VIDEO_PATH, start, end, out_path)
    if ok:
        n_ok += 1
    else:
        print(f'  FAIL: {out_name}')
        n_fail += 1

print(f'\nDone: {n_ok} clips saved, {n_fail} failed')
print(f'Output → {OUTPUT_DIR}')

## 10. (Optional) Extract frames from clips for annotation

Extract frames at a fixed fps from the closeup clips — use as training data input.

In [ ]:
FRAME_FPS  = 2.0
FRAMES_DIR = OUTPUT_DIR / 'frames'
FRAMES_DIR.mkdir(exist_ok=True)

n_frames = 0
cap  = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
step = max(1, int(round(fps / FRAME_FPS)))

for i, (start, end, _) in enumerate(tqdm(clips, desc='extracting frames')):
    start_f = int(start * fps)
    end_f   = int(end   * fps)
    for idx in range(start_f, end_f, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        fname = FRAMES_DIR / f'clip{i+1:03d}_{idx:07d}.jpg'
        cv2.imwrite(str(fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        n_frames += 1

cap.release()
print(f'Extracted {n_frames} frames → {FRAMES_DIR}')